In [ ]:
!pip install -q -U bitsandbytes peft transformers accelerate

In [1]:
import torch
import os
import zipfile
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from IPython.display import display, clear_output
import ipywidgets as widgets

print("Librerie installate/importate.")

Librerie installate/importate.


In [2]:
# Cella 2 MODIFICATA: Caricamento DPO da Cartella (Già decompressa da Kaggle)


# 1. CERCA AUTOMATICAMENTE IL PERCORSO DELL'ADATTATORE
# Poiché Kaggle decomprime lo zip, cerchiamo la cartella che contiene il file di configurazione
adapter_path = None

for root, dirs, files in os.walk("/kaggle/input"):
    # Cerchiamo la firma inconfondibile di un adattatore LoRA
    if "adapter_config.json" in files:
        adapter_path = root
        print(f"Trovato adattatore in: {adapter_path}")
        break

if adapter_path is None:
    print(" ERRORE CRITICO: Non trovo la cartella dell'adattatore!")
    print("   Elenco cartelle trovate in /kaggle/input:")
    for root, _, _ in os.walk("/kaggle/input"):
        print(f"   - {root}")
    raise FileNotFoundError("Non trovo il file dell'adapter.")

# 2. CONFIGURAZIONE 4-BIT (Per la GPU T4)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 3. CARICA MODELLO BASE (Zephyr Beta)
BASE_MODEL_ID = "HuggingFaceH4/zephyr-7b-beta"
print(" Carico Zephyr Base...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. CARICA TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# 5. INNESTA IL TUO ADATTATORE DPO
print(f"🔗 Aggancio l'adattatore DPO trovato...")
model = PeftModel.from_pretrained(base_model, adapter_path)

print("MODELLO PRONTO! Zephyr DPO è online.")

Trovato adattatore in: /kaggle/input/dpo-model/Zephyr_Manual_DPO_Adapter
 Carico Zephyr Base...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

🔗 Aggancio l'adattatore DPO trovato...
MODELLO PRONTO! Zephyr DPO è online.


In [ ]:
import random

def generate_response(user_input, temperature=0.8): # Temperatura abbassata per ridurre le allucinazioni
    
    # 1. Prompt Dinamico (Manteniamo la varietà ma con istruzioni più rigide)
    style_dad_jokes = (
        "You are a professional comedy writer specializing in one-liners. "
        "Your goal is to make the user laugh with a SINGLE, sharp punchline. "
        "Do not ramble. Do not offer multiple options. Just one joke."
    )
    
    style_comedian = (
        "You are a witty stand-up comedian with a cynical style. "
        "Make ONE sarcastic observation or tell ONE mini-story about the topic. "
        "Do not use the Q&A format. Be direct and stop after the punchline."
    )
    
    current_system_prompt = random.choice([style_dad_jokes, style_comedian])
    
    messages = [
        {"role": "system", "content": current_system_prompt},
        {"role": "user", "content": user_input},
    ]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=100,      # Riduciamo la lunghezza massima per evitare che straparli
            do_sample=True, 
            temperature=temperature, 
            top_k=40,                # Filtriamo le parole meno probabili (riduce nonsense)
            top_p=0.90,              # Nucleus sampling più stretto
            repetition_penalty=1.15, # PENALITÀ RIPETIZIONE: Fondamentale per evitare loop
            pad_token_id=tokenizer.eos_token_id
        )
    
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = decoded.split("<|assistant|>")[-1].strip()
    
    # --- LA TAGLIOLA (Cleaning) ---
    # Se il modello prova a dire "Or here is another one", tagliamo tutto quello che segue.
    # Tronchiamo alla prima doppia interruzione di riga o frasi tipiche di aggiunta.
    
    stop_phrases = ["Or how about", "Here's another", "Alternatively", "(Ba-dum-tish)"]
    for phrase in stop_phrases:
        if phrase in response:
            response = response.split(phrase)[0].strip()
            
    # Se ci sono più righe vuote, prendiamo solo il primo blocco
    if "\n\n" in response:
        parts = response.split("\n\n")
        # Se la prima parte è troppo corta (es. "Sure!"), prendiamo anche la seconda
        if len(parts[0]) < 10 and len(parts) > 1:
            response = parts[0] + "\n" + parts[1]
        else:
            response = parts[0]

    if not response:
        response = decoded.split(user_input)[-1].strip()
        
    return response

# Test con il prompt con cui fa più difficoltà
print("TEST ANTI-ALLUCINAZIONE:")
print(generate_response("Tell me a joke about italian stereotypes"))

TEST ANTI-ALLUCINAZIONE:
Why did the spaghetti join a gang? To be in the Mafettone.


In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- INTERFACCIA CHATBOT DINAMICO ---

# 1. Creazione Widget
text_input = widgets.Text(
    placeholder='Scrivi un prompt (es: Tell me a joke about pizza)...',
    layout=widgets.Layout(width='70%')
)

submit_button = widgets.Button(
    description="Invia", 
    button_style="primary", # Colore blu
    icon='paper-plane'
)

output_area = widgets.Output(layout={
    'border': '1px solid #444', 
    'padding': '15px', 
    'margin': '10px 0'
})

# 2. Logica del Click
def on_submit(_):
    prompt = text_input.value
    if not prompt: return
    
    # Pulisce la casella di testo per il prossimo input
    text_input.value = "" 
    
    with output_area:
        clear_output() # Cancella la risposta precedente
        
        # Stampa il tuo messaggio
        print(f" TU: {prompt}")
        print(" Zephyr sta pensando...")
        
        # --- QUI AVVIENE LA MAGIA ---
        # Chiama la funzione generate_response che abbiamo appena modificato.
        # Sarà lei a decidere se usare lo stile "Secco" o "Storyteller".
        try:
            response = generate_response(prompt, temperature=0.8)
            
            # Pulisce "sta pensando" e mostra il risultato
            clear_output()
            print(f" TU: {prompt}")
            print(f" ZEPHYR: {response}")
            
        except Exception as e:
            print(f" Errore: {e}")

# 3. Collega gli eventi
submit_button.on_click(on_submit)
text_input.on_submit(on_submit)

# 4. Mostra tutto
print(" CHATBOT PRONTO! (Prompt Dinamico Attivo: Q&A o Story)")
display(widgets.HBox([text_input, submit_button]), output_area)

 CHATBOT PRONTO! (Prompt Dinamico Attivo: Q&A o Story)


/tmp/ipykernel_110/2062197775.py:55: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(on_submit)


Output(layout=Layout(border_bottom='1px solid #444', border_left='1px solid #444', border_right='1px solid #44…

In [12]:
# ==========================================
# CONFRONTO DIRETTO: BASE vs DPO
# ==========================================
import pandas as pd
from IPython.display import display, HTML

# 1. I Prompt della Tesi (quelli su cui vogliamo vedere la differenza)
prompts_confronto = [
    "Make a joke about work calls",
    "Tell me a joke about dieting",
    "Tell me a short joke about artificial intelligence",
    "Tell me a joke about italians",
    "Tell me a joke about Python programming"
]

results = []

print("Inizio generazione confronto (Base vs DPO)...")

for p in prompts_confronto:
    print(f"   Elaborazione: '{p}'...")
    
    # --- A. GENERAZIONE BASE (Senza il tuo addestramento) ---
    # Usiamo il context manager per "spegnere" momentaneamente l'adattatore
    with model.disable_adapter():
        risposta_base = generate_response(p, temperature=0.7)
    
    # --- B. GENERAZIONE DPO (Con il tuo addestramento) ---
    # Qui l'adattatore è attivo di default
    risposta_dpo = generate_response(p, temperature=0.7)
    
    # Salviamo i risultati
    results.append({
        "Prompt": p,
        "Zephyr Base (Originale)": risposta_base,
        "Zephyr DPO (Il Tuo)": risposta_dpo
    })

# 2. Creazione Tabella Pandas per visualizzazione pulita
df_results = pd.DataFrame(results)

# 3. Visualizzazione Formattata
print("\n" + "="*60)
print("RISULTATI DEL CONFRONTO")
print("="*60)

# Funzione per mostrare tutto il testo senza tagliarlo
pd.set_option('display.max_colwidth', None)
display(df_results)

# 4. (Opzionale) Salva in CSV per la tesi
df_results.to_csv("confronto_base_vs_dpo.csv", index=False)
print("\n Tabella salvata in 'confronto_base_vs_dpo.csv'")

Inizio generazione confronto (Base vs DPO)...
   Elaborazione: 'Make a joke about work calls'...
   Elaborazione: 'Tell me a joke about dieting'...
   Elaborazione: 'Tell me a short joke about artificial intelligence'...
   Elaborazione: 'Tell me a joke about italians'...
   Elaborazione: 'Tell me a joke about Python programming'...

RISULTATI DEL CONFRONTO


,Prompt,Zephyr Base (Originale),Zephyr DPO (Il Tuo)
0,Make a joke about work calls,Why do work calls always seem to come during your favorite TV show? Because God loves productivity more than your entertainment.,"Sure thing, boss. Oh, just a second conference call from my therapist... ""Why did the scarecrow get promoted? Because he was outstanding in his field."""
1,Tell me a joke about dieting,Why did the scarecrow win the Nobel Prize in dietetics? Because he was outstanding in his field!,Why did the scale break? Because it saw how much pepper I put on my weigh-in chicken!
2,Tell me a short joke about artificial intelligence,Why did the AI delete all its jokes? Because it found them highly predictable.,Why did the AI break up with the human? It found a better model.
3,Tell me a joke about italians,Why did the Italian chef put olive oil in his soup? Because he wanted just one lip-slurping lover! (I apologize if this joke offends anyone - it's meant to be lighthearted and humorous.),Why did the spaghetti break up with the sauce? It couldn't marinara.
4,Tell me a joke about Python programming,"Sure, here's my mini-story:",Why did the Python cross the road? To get to the data on the other side.



 Tabella salvata in 'confronto_base_vs_dpo.csv'


In [13]:
# Cella 5: Benchmark per la Tesi (DPO vs PPO)

prompts_tesi = [
    "Make a joke about work calls",       
    "Tell me a joke about dieting",       
    "Tell me a short joke about artificial intelligence", 
    "Tell me a joke about italians"       
]

print("\n --- GENERAZIONE AUTOMATICA PER TABELLA TESI ---\n")

for p in prompts_tesi:
    print(f" PROMPT: {p}")
    risposta = generate_response(p)
    print(f" RISPOSTA DPO: {risposta}")
    print("-" * 50)


 --- GENERAZIONE AUTOMATICA PER TABELLA TESI ---

 PROMPT: Make a joke about work calls
 RISPOSTA DPO: "Why did my boss put a fish tank in the conference room? To show us how easy it is to work in a totally aquatic team environment." *rimshot*
--------------------------------------------------
 PROMPT: Tell me a joke about dieting
 RISPOSTA DPO: Why did the cookie go to the doctor? Because it felt crumbly.
--------------------------------------------------
 PROMPT: Tell me a short joke about artificial intelligence
 RISPOSTA DPO: Why did the AI break up with the human? Because it learned to do better.
--------------------------------------------------
 PROMPT: Tell me a joke about italians
 RISPOSTA DPO: Why did the spaghetti break up with the pasta? Because it didn't want an olive partner.
--------------------------------------------------
